# 03 - Smoke-100 Training (Colab + Kaggle)
100 files, leakage-safe Pipeline, XGBoost + Optuna (5 trials), MLflow, SHAP check.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
ON_KAGGLE = os.path.isdir('/kaggle/working')
ON_COLAB = os.path.isdir('/content/drive')
ROOT = '/kaggle/working/repo' if ON_KAGGLE else ('/content/drive/MyDrive/DeepFakeVoiceResearch' if ON_COLAB else os.getcwd())
print(f'Kaggle={ON_KAGGLE} Colab={ON_COLAB} root={ROOT}')
if ROOT not in sys.path: sys.path.insert(0, ROOT)
%cd $ROOT

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import yaml
with open(os.path.join(ROOT,'configs/features.yaml')) as f: feat_cfg=yaml.safe_load(f)
with open(os.path.join(ROOT,'configs/models.yaml')) as f: model_cfg=yaml.safe_load(f)
WORK = '/kaggle/working' if ON_KAGGLE else ROOT
OUT_MODELS = os.path.join(WORK,'models'); OUT_RES=os.path.join(WORK,'results')
os.makedirs(OUT_MODELS,exist_ok=True); os.makedirs(OUT_RES,exist_ok=True)
if ON_KAGGLE:
    feat_cfg['features']['cqcc']['use_cached']=False  # avoid Drive dep; dim=278
    model_cfg['mlflow']['tracking_uri']=os.path.join(WORK,'mlruns')
print('use_cached:',feat_cfg['features']['cqcc']['use_cached'])


In [ ]:
import glob, librosa, numpy as np, os
from src.features.mfcc import extract_mfcc
from src.features.lfcc import extract_lfcc
from src.features.spectral import extract_spectral
from src.features.fusion import fuse_features_for_file
from src.utils.cache_loader import load_cached_cqcc

EXT = {'mfcc': extract_mfcc, 'lfcc': extract_lfcc, 'spectral': extract_spectral}

# Pointing to the correct smoke_test folder
smoke_dir = os.path.join(ROOT, 'datasets/smoke_test')
bona = sorted(glob.glob(os.path.join(smoke_dir, 'bonafide/*.flac')))[:50]
spoof = sorted(glob.glob(os.path.join(smoke_dir, 'spoof/*.flac')))[:50]

files = bona + spoof
labels = [0] * len(bona) + [1] * len(spoof)

cache = {}
cache_path = os.path.join(smoke_dir, 'cqcc_cache_smoke.h5')
if feat_cfg['features']['cqcc']['use_cached'] and os.path.exists(cache_path):
    cache = load_cached_cqcc(cache_path, [os.path.basename(p) for p in files])

print(f'files={len(files)} cache_hits={len(cache)}')

X, y = [], []
for p, l in zip(files, labels):
    a, sr = librosa.load(p, sr=feat_cfg['sample_rate'])
    v = fuse_features_for_file(a, sr, feat_cfg, EXT, cqcc_mat=cache.get(os.path.basename(p)))
    X.append(v)
    y.append(l)

X = np.array(X)
y = np.array(y)
print('X', X.shape, 'NaNs', int(np.isnan(X).sum()))

In [ ]:
import src.models.train_xgb as train_mod
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold, SelectKBest, mutual_info_classif

def fixed_build_pipeline(cfg, n_feat, params=None):
    # Safely parse and enforce float conversion regardless of yaml type
    raw_vt = cfg.get("feature_select", {}).get("variance_threshold", 1e-4)
    vt = float(raw_vt) if raw_vt is not None else 1e-4
    
    raw_k = cfg.get("feature_select", {}).get("mutual_info_k", 200)
    k = max(1, min(int(raw_k), int(n_feat)))
    
    rs = cfg.get("training", {}).get("random_state", 42)
    clf_params = dict(
        n_estimators=200, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
        eval_metric="logloss", tree_method="hist",
        random_state=rs, n_jobs=-1,
    )
    if params:
        clf_params.update(params)
    XGBClassifier = train_mod._xgb_class()
    return Pipeline([
        ("scaler", StandardScaler()),
        ("var", VarianceThreshold(threshold=vt)),
        ("select", SelectKBest(mutual_info_classif, k=k)),
        ("clf", XGBClassifier(**clf_params)),
    ])

# Apply patch
train_mod.build_pipeline = fixed_build_pipeline
print("✅ Pipeline builder successfully patched with explicit float casting.")

In [ ]:
from src.models.train_xgb import tune_and_train
import json
best,metrics,path=tune_and_train(X,y,feat_cfg,model_cfg,n_trials=5,output_dir=OUT_MODELS,experiment='smoke')
print(metrics); print('saved',path)
open(os.path.join(OUT_RES,'smoke_metrics.json'),'w').write(json.dumps(metrics,indent=2))


In [ ]:
import shap, numpy as np

# Transform data using all pipeline steps except the final classifier
X_transformed = m[:-1].transform(X[:10])

# Now initialize TreeExplainer on the transformed data
explainer = shap.TreeExplainer(m.named_steps['clf'])
shap_values = explainer.shap_values(X_transformed)
print('SHAP ok', np.shape(shap_values))